In [12]:
import os
import subprocess
import zipfile
import hashlib
import json
import urllib.request
import pandas as pd

# 1. Define Beocat Paths
HOME_DIR = os.path.expanduser("~/frauddetection")
PROJECT_DIR = os.path.join(HOME_DIR, "ieee_fraud")
BRONZE_RAW = os.path.join(PROJECT_DIR, "bronze", "raw")
SILVER_PATH = os.path.join(PROJECT_DIR, "silver", "transactions_cleaned.parquet")

os.makedirs(BRONZE_RAW, exist_ok=True)
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)

print(f"Project Workspace: {PROJECT_DIR}")

# 2. Configure Kaggle Environment Variable from your access_token file
token_path = os.path.expanduser("~/.kaggle/access_token")
if os.path.exists(token_path):
    with open(token_path, "r") as f:
        token = f.read().strip()
    os.environ["KAGGLE_API_TOKEN"] = token
else:
    token = os.getenv("KAGGLE_API_TOKEN")

if not token:
    raise ValueError("Kaggle API token not found in ~/.kaggle/access_token or environment variables.")

# 3. Check/List Competition Files and Store the Result
log_file_path = os.path.join(PROJECT_DIR, "kaggle_files_check.log")

result = subprocess.run(
    [
        "kaggle",
        "competitions",
        "files",
        "-c", "ieee-fraud-detection"
    ],
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", result.returncode)
print(result.stdout)

# Store the file check result into Beocat storage
with open(log_file_path, "w") as f:
    f.write(f"Return code: {result.returncode}\n")
    f.write("--- STDOUT ---\n")
    f.write(result.stdout)
    if result.returncode != 0:
        f.write("\n--- STDERR ---\n")
        f.write(result.stderr)

print(f"Command output stored at: {log_file_path}")

if result.returncode != 0:
    raise RuntimeError(f"Kaggle CLI error: {result.stderr}")

# 4. Download the Fraud-Detection Dataset ZIP into Bronze Raw
zip_path = os.path.join(BRONZE_RAW, "ieee-fraud-detection.zip")
archive_extracted_marker = os.path.join(BRONZE_RAW, ".extracted")

if not os.path.exists(archive_extracted_marker) and not os.path.exists(zip_path):
    print("Downloading IEEE-CIS Fraud Detection dataset...")
    
    download_result = subprocess.run(
        [
            "kaggle", "competitions", "download",
            "-c", "ieee-fraud-detection",
            "-p", BRONZE_RAW
        ],
        capture_output=True,
        text=True,
        timeout=600
    )
    
    if download_result.returncode != 0:
        print(download_result.stderr)
        raise RuntimeError("Kaggle download failed.")
    print("Download complete.")
else:
    print("Dataset ZIP archive already exists or is already extracted.")

# 5. Expected CSV Files for the Competition
expected_files = [
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
    "sample_submission.csv",
]

# Extract CSVs safely into Bronze raw directory
print("Extracting archive contents...")
with zipfile.ZipFile(zip_path, "r") as z:
    names = set(z.namelist())
    missing = [f for f in expected_files if f not in names]
    if missing:
        raise ValueError(f"Missing files in ZIP: {missing}")
    
    for filename in expected_files:
        destination = os.path.join(BRONZE_RAW, filename)
        if not os.path.exists(destination):
            print(f"Extracting: {filename}")
            z.extract(filename, BRONZE_RAW)
        else:
            print(f"Already exists; skipping: {filename}")

# Create marker so we don't re-extract every time
with open(archive_extracted_marker, "w") as f:
    f.write("extracted")

# 6. Compute SHA-256 Checksums to Validate Stored Bronze Data
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("\n--- Bronze Layer Integrity Check ---")
for filename in expected_files:
    path = os.path.join(BRONZE_RAW, filename)
    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"{filename}: {size_mb:.2f} MB | SHA-256: {sha256_file(path)}")

print("\nPASS: All fraud-detection data successfully stored and structured in Bronze layer.")

Project Workspace: /homes/sekabanj/frauddetection/ieee_fraud
Return code: 0
name                         size  creationDate                
---------------------  ----------  --------------------------  
sample_submission.csv     6080314  2019-12-11 23:08:51.100000  
test_identity.csv        25797161  2019-12-11 23:08:51.619000  
test_transaction.csv    613194934  2019-12-11 23:09:18.466000  
train_identity.csv       26529680  2019-12-11 23:09:00.658000  
train_transaction.csv   683351067  2019-12-11 23:09:14.423000  

Command output stored at: /homes/sekabanj/frauddetection/ieee_fraud/kaggle_files_check.log
Download complete.
Extracting archive contents...
Extracting: train_transaction.csv
Extracting: train_identity.csv
Extracting: test_transaction.csv
Extracting: test_identity.csv
Extracting: sample_submission.csv

--- Bronze Layer Integrity Check ---
train_transaction.csv: 651.69 MB | SHA-256: 3a5c83ab6b3cc13dcabe5ffa9f522307fd5f7f7b6e6f6a60c32284ca6283d642
train_identity.csv: 25.30